# RETINA-AI: Veri Yukleme ve On Isleme Rehberi (Ortak Sablon)

Bu notebook, **Firat Universitesi Bilgisayar Muhendisligi Medikal Goruntu Isleme** dersi kapsaminda gelistirilen RETINA-AI projesinin temel veri yukleme ve on isleme hattini icerir.

Tum ekip uyeleri (Kisi 2, 3, 4, 5) calismalarina baslarken verileri bu notebook uzerinden temin edebilir ve `on_isle()` fonksiyonunu ortak standart olarak kullanabilir.

---
### Icerik:
1. Calisma Ortaminin Hazirlanmasi (Drive Baglantisi ve Yol Ayarlari)
2. Veri Bolme Tablosunun (`veri_bolme.csv`) Yuklenmesi
3. Sinif ve Bolum Dagilimlarinin Incelenmesi
4. `on_isle()` Pipeline Fonksiyonunun Calistirilmasi
5. Diger Moduller Icin Ornek Veri Erisim Sablonu (Batch / Dataset)

## 1. Calisma Ortaminin Hazirlanmasi
Google Colab ortaminda calisirken Google Drive baglantisi kurulabilir. Yerel ortamda calisiyorsaniz bu hucre otomatik olarak yerel yolu tespit eder.

In [ ]:
# Ortam kontrolu ve yol tanimlamalari
import sys
from pathlib import Path

try:
    from google.colab import drive
    drive.mount('/content/drive')
    proje_koku = Path('/content/drive/MyDrive/retina-ai-ocular-disease-classification')
    print('[BILGI] Colab Drive modu aktif.')
except ImportError:
    proje_koku = Path('.').resolve().parent if Path('.').resolve().name == 'notebooks' else Path('.').resolve()
    print(f'[BILGI] Yerel calisma ortami aktif: {proje_koku}')

if str(proje_koku) not in sys.path:
    sys.path.append(str(proje_koku))


## 2. Veri Bolme Tablosunun (`veri_bolme.csv`) Yuklenmesi
Toplam 4028 gorsel, her siniftan 1007 adet olacak sekilde %70 Train (705), %15 Val (151) ve %15 Test (151) olarak ayrilmistir.

In [ ]:
import pandas as pd

csv_yolu = proje_koku / 'data' / 'splits' / 'veri_bolme.csv'
df = pd.read_csv(csv_yolu)

print(f'Toplam ornek sayisi: {len(df)}')
df.head()

In [ ]:
# Sinif ve kume dagilimi
dagilim_tablosu = pd.crosstab(df['class_name'], df['split'], margins=True)
print('--- VERI SETI DAGILIM TABLOSU ---')
display(dagilim_tablosu) if 'display' in globals() else print(dagilim_tablosu)


## 3. `on_isle()` Fonksiyonunun Test Edilmesi
Bu fonksiyon sirasiyla:
- Otomatik ROI kirpma & dairesel maskeleme
- 512x512 boyutlandirma
- Yesil kanal izolasyonu
- CLAHE kontrast esitleme (clipLimit=2.0, tileGridSize=(8, 8))
- 3x3 Median filtre ile gurultu azaltma
- [0.0, 1.0] normalizasyonu uygular.

In [ ]:
import cv2
import matplotlib.pyplot as plt
from src.preprocessing import on_isle

siniflar = sorted(df['class_name'].unique())
fig, axes = plt.subplots(len(siniflar), 2, figsize=(8, 12))

for idx, sinif in enumerate(siniflar):
    ornek_satir = df[df['class_name'] == sinif].iloc[0]
    dosya_yolu = proje_koku / ornek_satir['relative_path'] if not Path(ornek_satir['file_path']).exists() else Path(ornek_satir['file_path'])
    
    # Orijinal gorsel
    orijinal = cv2.imread(str(dosya_yolu))
    orijinal_rgb = cv2.cvtColor(orijinal, cv2.COLOR_BGR2RGB)
    
    # on_isle()
    islenmis = on_isle(dosya_yolu)
    
    axes[idx, 0].imshow(orijinal_rgb)
    axes[idx, 0].set_title(f'{sinif.upper()} - Orijinal')
    axes[idx, 0].axis('off')
    
    axes[idx, 1].imshow(islenmis, cmap='gray')
    axes[idx, 1].set_title(f'{sinif.upper()} - on_isle() Ciktisi')
    axes[idx, 1].axis('off')

plt.tight_layout()
plt.show()


## 4. Ekipler Icin Veri Erisim Fonksiyonu
Kisi 2 (Damar/Disk), Kisi 3 (Oznitelik/SVM) ve Kisi 4 (Derin Ogrenme) istedikleri bolumun (train/val/test) dosya yollarini ve etiketlerini asagidaki gibi kolayca alabilir.

In [ ]:
def veri_kumesini_al(bolum='train'):
    """
    Secilen bolumdeki (train, val, test) gorsel dosya yollarini ve siniflarini dondurur.
    """
    alt_df = df[df['split'] == bolum].reset_index(drop=True)
    dosya_listesi = alt_df['file_path'].tolist()
    etiket_listesi = alt_df['class_name'].tolist()
    return dosya_listesi, etiket_listesi

train_yollar, train_etiketler = veri_kumesini_al('train')
val_yollar, val_etiketler = veri_kumesini_al('val')
test_yollar, test_etiketler = veri_kumesini_al('test')

print(f'Egitim Kumesi: {len(train_yollar)} ornek')
print(f'Dogrulama Kumesi: {len(val_yollar)} ornek')
print(f'Test Kumesi: {len(test_yollar)} ornek')
